# From Cell Ranger output to denoised counts

The shortest path from a Cell Ranger run to a denoised, analysis-ready AnnData.

**What you need:**

- Cell Ranger's **filtered** matrix (defines your cells).
- The matching **raw** matrix from the same library (defines the empty-droplet pool used to estimate ambient RNA).
- Raw integer UMI counts — not normalized, log-transformed, or HVG-filtered.

The documentation build does not execute this notebook; it requires local input files. Fill in the two paths below and run the cells in order.

In [ ]:
import scanpy as sc
import ambidose as amdose

## 1. Point at your Cell Ranger `outs/`

In [ ]:
FILTERED = "/data/project/sample_01/outs/filtered_feature_bc_matrix"
RAW = "/data/project/sample_01/outs/raw_feature_bc_matrix.h5"

## 2. Load your filtered matrix the usual scanpy way

Use `sc.read_10x_mtx` for an MTX directory or `sc.read_10x_h5` for an HDF5 file — whichever matches your `FILTERED` path.

In [ ]:
adata = sc.read_10x_mtx(FILTERED)
adata

## 3. Denoise

`raw=` tells AmbiDose where to find the matching empty-droplet pool. `adata`'s own barcodes become the cell whitelist. This returns a new, denoised copy of `adata` — it does not mutate the object you passed in.

In [ ]:
adata = amdose.denoise(adata, raw=RAW, sample_key=None, report=True)

If you already have a trusted broad cell-type annotation, pass it as `type_key="your_column"` (it must exist in `adata.obs` and label every cell) instead of letting `denoise()` build automatic Leiden groups.

For multiple GEM wells loaded into one `adata`, use a library/GEM column as `sample_key=` instead of `None` — see {doc}`../user_guide/workflow` for the multi-sample data contract.

`report=True` writes a QC HTML report (`ambidose_report.html` in the current directory by default; pass a path instead to choose one) from the raw pool before it's reduced to `adata`'s own cells, so it still gets the full empty-droplet panels — the path used is printed to stderr.

Progress is printed to stderr as each stage runs (ambient profile, coarse typing, dose, subtraction); a real dataset can take a few minutes.

**Performance parameters.** AmbiDose is CPU-only; two arguments to `denoise()` control how it uses the machine, and the defaults are usually right, but are worth knowing about on a shared or scheduled system:

- `n_jobs=` caps the workers used for Scanpy's KNN/Leiden graph, the cross-cell structure-regression blocks, and the Monte Carlo p-value step in cell calling. It defaults to `None`, which auto-detects from CPU affinity and available RAM — pass an explicit integer (e.g. `n_jobs=4`) on a shared machine or inside a batch-scheduler job with a fixed core allocation.
- `typing_fast=` (default `True`) switches to a cheaper neighbor graph once a library has at least 20,000 cells, trading a small amount of clustering resolution for speed. Pass `typing_fast=False` (CLI `--full-typing`) for the full-cell graph on a large library if you have the time budget and want the more careful clustering.

Neither argument changes what gets removed — both only change how fast the run gets there.

```python
adata = amdose.denoise(adata, raw=RAW, sample_key=None, report=True, n_jobs=4)
```

## 4. Check the output

In [ ]:
print(adata.obs[["ambidose_rho", "ambidose_d"]].describe())
print(adata.obs["ambidose_rho_trust"].value_counts())
print(adata.layers["ambidose_denoised"])

| Field | Meaning |
|---|---|
| `X` | Denoised, non-negative integer UMI counts (`denoise()`'s default for both calling forms) |
| `layers["raw_counts"]` | The original input counts, preserved |
| `obs["ambidose_rho"]` | Per-cell ambient fraction |
| `obs["ambidose_d"]` | Per-cell absolute ambient UMI dose |
| `obs["ambidose_rho_trust"]` | `ok`, `low_evidence`, `ceiling_risk`, `type_structure_risk`, `under_execution`, or `over_removal` — review before treating `rho` as a quantitative contamination rate |
| `obs["ambidose_cluster"]` | Automatic label-free Leiden groups, when no `type_key` was given |

## 5. Continue with scanpy

In [ ]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

## 6. Save

In [ ]:
adata.write_h5ad("/data/project/sample_01/ambidose_cells.h5ad", compression="gzip")

## Command-line equivalent

`--input` accepts a Cell Ranger `outs/` directory directly and auto-pairs the raw matrix with the filtered barcodes:

```bash
ambidose denoise \
  --input /data/project/sample_01/outs \
  --report /data/project/sample_01/ambidose_report.html \
  --cells-only \
  --output /data/project/sample_01/ambidose_cells.h5ad
```

## Multiple libraries

Each independently prepared GEM well needs its own ambient profile — do not merge raw droplets across libraries before estimating it. For a directory of Cell Ranger sample folders:

```bash
ambidose denoise --root /data/project/runs \
  --sample-key sample \
  --output /data/project/ambidose_all_samples.h5ad
```

For HDF5 inputs, external whitelists, or per-library detail, see {doc}`../user_guide/cli` and {doc}`../user_guide/workflow`.

## If something looks wrong

| Symptom | Likely cause | Action |
|---|---|---|
| `0/N cell_barcodes matched` | `raw`/filtered mismatch | Confirm both paths are from the same Cell Ranger run |
| `Input must be raw integer UMI counts` | Data were normalized or log-transformed already | Reload the raw count matrix |
| `Fewer than 10 empty droplets` | Filtered (not raw) matrix loaded as `raw=` | Point `raw=` at the raw, unfiltered matrix |
| `AnnData view rejected` | A slice was passed directly | Call `.copy()` on the slice first |
| High fallback fraction / unusually high `rho` | Broadly expressed genes bias calibration on some tissues | Read {doc}`../user_guide/method`'s limitations section before trusting `rho` quantitatively |

For the fully manual path — loading the raw matrix yourself, inspecting empty droplets and $\chi$ step by step, or supplying an external cell whitelist (CellBender, EmptyDrops) — see {doc}`external_cell_whitelists` and {doc}`../user_guide/workflow`.